#Model Training
##1.1 Import Data and Required Packages
Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [2]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
!pip install catboost
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


####Import the Data as Pandas DataFrame

In [3]:
df = pd.read_parquet('/content/data_transformed.parquet')

#####Show Top 5 Records

In [4]:
df.head()

,Cement,Blast_Furnace_Slag,Fly_Ash,Water,Superplasticizer,Coarse_Aggregate,Fine_Aggregate,Age,Concrete_Strength,Water_Binder_Ratio,Log_Age,Cement_x_Age,SCM_Ratio
0,336.0,0.000000,0.000000,182.0,1.386294,986.0,817.0,28,44.864203,0.541667,3.367296,9408.0,0.000000
1,140.1,1.648659,5.379436,193.9,1.740466,1049.5,710.1,28,26.420000,0.538312,3.367296,3922.8,0.611049
2,350.0,0.000000,0.000000,203.0,0.000000,974.0,775.0,14,22.532076,0.580000,2.708050,4900.0,0.000000
3,162.0,5.003946,5.252273,179.0,2.995732,838.0,741.0,28,42.080000,0.358000,3.367296,4536.0,0.676000
4,225.0,0.000000,0.000000,181.0,0.000000,1113.0,833.0,7,11.169511,0.804444,2.079442,1575.0,0.000000


#####Preparing X and Y variables

In [6]:
X = df.drop(columns=['Concrete_Strength'],axis=1)

In [7]:
X.head()

,Cement,Blast_Furnace_Slag,Fly_Ash,Water,Superplasticizer,Coarse_Aggregate,Fine_Aggregate,Age,Water_Binder_Ratio,Log_Age,Cement_x_Age,SCM_Ratio
0,336.0,0.000000,0.000000,182.0,1.386294,986.0,817.0,28,0.541667,3.367296,9408.0,0.000000
1,140.1,1.648659,5.379436,193.9,1.740466,1049.5,710.1,28,0.538312,3.367296,3922.8,0.611049
2,350.0,0.000000,0.000000,203.0,0.000000,974.0,775.0,14,0.580000,2.708050,4900.0,0.000000
3,162.0,5.003946,5.252273,179.0,2.995732,838.0,741.0,28,0.358000,3.367296,4536.0,0.676000
4,225.0,0.000000,0.000000,181.0,0.000000,1113.0,833.0,7,0.804444,2.079442,1575.0,0.000000


In [9]:
y = df['Concrete_Strength']

In [10]:
y

,Concrete_Strength
0,44.864203
1,26.420000
2,22.532076
3,42.080000
4,11.169511
...,...
1128,67.568648
1129,43.499041
1130,18.415904
1131,19.691435


Standard scaling was applied to all numerical features after log transformation to ensure uniform feature scale, improving numerical stability for linear models.

In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X.shape

(1133, 12)

In [14]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((906, 12), (227, 12))

####Create an Evaluate Function to give all metrics after model Training

In [15]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [16]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

In [17]:
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)


    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')

    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)

    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 6.7029
- Mean Absolute Error: 5.2033
- R2 Score: 0.8248
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 7.1692
- Mean Absolute Error: 5.6178
- R2 Score: 0.8087


Lasso
Model performance for Training set
- Root Mean Squared Error: 7.4686
- Mean Absolute Error: 5.7814
- R2 Score: 0.7825
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 7.7856
- Mean Absolute Error: 6.0395
- R2 Score: 0.7744


Ridge
Model performance for Training set
- Root Mean Squared Error: 6.7495
- Mean Absolute Error: 5.2326
- R2 Score: 0.8224
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 7.1599
- Mean Absolute Error: 5.6132
- R2 Score: 0.8092


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 8.1311
- Mean Absolute Error: 6.1316
- R2 Score: 0.7422
-----------------------

####Results

In [18]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
7,CatBoosting Regressor,0.945232
6,XGBRegressor,0.929165
5,Random Forest Regressor,0.907312
4,Decision Tree,0.837775
8,AdaBoost Regressor,0.815898
2,Ridge,0.809231
0,Linear Regression,0.808732
1,Lasso,0.774427
3,K-Neighbors Regressor,0.594139


CatBoost Regressor achieved the highest R² score due to its ability to model complex non-linear relationships and feature interactions inherent in concrete mix design. Unlike linear models, it is robust to multicollinearity and outliers, and does not require feature scaling. Its gradient boosting framework iteratively improves predictions, making it well-suited for structured tabular data.